In [97]:
import pandas as pd
import numpy as np
import tools


In [98]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")


In [99]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")

df_sprzedaz = df_kalendarz[df_kalendarz['TypRuchu'] == 'sprzedaz'].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")

kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort',
                  'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwy']

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)


# Zakres życia per TowId (z bufor 14 dni,)
daty_sku = df_sprzedaz.groupby('TowId')['Data'].agg(DataStart='min', DataKoniec='max').reset_index()
daty_sku['DataKoniec'] = (daty_sku['DataKoniec'] + pd.Timedelta(days=14)).clip(upper=data_max)

# Dołączamy do pelny_kalendarz i filtrujemy
pelny_kalendarz = pelny_kalendarz.merge(daty_sku, on='TowId', how='left')

przed = len(pelny_kalendarz)
pelny_kalendarz = pelny_kalendarz[
    (pelny_kalendarz['Data'] >= pelny_kalendarz['DataStart']) &
    (pelny_kalendarz['Data'] <= pelny_kalendarz['DataKoniec'])
].copy()

print(f"Przed przycięciem: {przed:,} wierszy")
print(f"Po przycięciu:     {len(pelny_kalendarz):,} wierszy")

# Opcjonalnie: usuń pomocnicze kolumny DataStart/DataKoniec, jeśli nie chcesz ich w finalnym pliku
pelny_kalendarz = pelny_kalendarz.drop(columns=['DataStart', 'DataKoniec'])




print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")


TowId z jakąkolwiek sprzedażą: 12,481
Dni w zakresie: 1,127
Rozmiar siatki: 14,066,087
Rozmiar po scaleniu: 15,810,161
Przed przycięciem: 15,810,161 wierszy
Po przycięciu:     8,802,515 wierszy

Puste rekordy (DokId=-1): 5,428,540
Realne rekordy sprzedaży: 3,373,975


In [100]:
pelny_kalendarz.sample(5)

,TowId,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,NazwaAsort,Dokument,WplywNaStan,MetodaLiczenia,Mnoznik,TypRuchu,CzyNiechciane,NazwaTowCleanName,NazwaAsortCleanName,JestMartwy
7067766,55479,2025-12-24,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,AKCESORIA KUCHENNE /przemysłowe,NaN,NaN,NaN,NaN,NaN,NaN,papier do pieczenia 10szt stella,akcesoria kuchenne przemyslowe,False
2359581,54927,2024-02-11,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,MAKARONY /zbożowe i sypkie,NaN,NaN,NaN,NaN,NaN,NaN,makaron lazanki 400g lubella,makarony zbozowe i sypkie,False
14274010,80134,2025-10-02,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NAPOJE NIEGAZOWANE,NaN,NaN,NaN,NaN,NaN,NaN,napoj herb ziel melon 0 5l herbapol,napoje niegazowane,False
3478652,51895,2023-01-04,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NAPOJE NIEGAZOWANE,NaN,NaN,NaN,NaN,NaN,NaN,sok baby jabl ban marc 300ml maspex,napoje niegazowane,True
2719745,2213,2023-05-05,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,WODY,NaN,NaN,NaN,NaN,NaN,NaN,woda min n gaz kropla beskidu 1 5l coca cola,wody,False


In [101]:
# Uzupełnianie brakujące wartości w kolumnach
pelny_kalendarz['Kolejnosc'] = pelny_kalendarz['Kolejnosc'].fillna(0)
pelny_kalendarz['NrPozycji'] = pelny_kalendarz['NrPozycji'].fillna(0)
pelny_kalendarz['TypPoz'] = pelny_kalendarz['TypPoz'].fillna(4)
pelny_kalendarz['IloscPlus'] = pelny_kalendarz['IloscPlus'].fillna(0)
pelny_kalendarz['IloscMinus'] = pelny_kalendarz['IloscMinus'].fillna(0)
pelny_kalendarz['CenaPoRab'] = pelny_kalendarz['CenaPoRab'].ffill()
pelny_kalendarz['Wartosc'] = pelny_kalendarz['Wartosc'].fillna(0)
pelny_kalendarz['KolejnyWDniu'] = pelny_kalendarz['KolejnyWDniu'].fillna(-1)
pelny_kalendarz['NrDok'] = pelny_kalendarz['NrDok'].fillna('brak')
pelny_kalendarz['TypDok'] = pelny_kalendarz['TypDok'].fillna(21)
pelny_kalendarz['AktywnyDok'] = pelny_kalendarz['AktywnyDok'].fillna(1)
pelny_kalendarz['Razem'] = pelny_kalendarz['Razem'].fillna(0)
pelny_kalendarz['DoZaplaty'] = pelny_kalendarz['DoZaplaty'].fillna(0)
pelny_kalendarz['Zaplacono'] = pelny_kalendarz['Zaplacono'].fillna(0)
pelny_kalendarz['Opis1'] = pelny_kalendarz['Opis1'].fillna('brak')
pelny_kalendarz['Producent'] = pelny_kalendarz['Producent'].fillna(-1)
pelny_kalendarz['AktywnyTow'] = pelny_kalendarz['AktywnyTow'].fillna(1)
pelny_kalendarz['Dokument'] = pelny_kalendarz['Dokument'].fillna("DF")
pelny_kalendarz['WplywNaStan'] = pelny_kalendarz['WplywNaStan'].fillna(False).astype(int)
pelny_kalendarz['MetodaLiczenia'] = pelny_kalendarz['MetodaLiczenia'].fillna('IP')
pelny_kalendarz['Mnoznik'] = pelny_kalendarz['Mnoznik'].fillna(0)
pelny_kalendarz['TypRuchu'] = pelny_kalendarz['TypRuchu'].fillna('sprzedaz')
pelny_kalendarz['CzyNiechciane'] = pelny_kalendarz['CzyNiechciane'].fillna(False).astype(int)
pelny_kalendarz['JestMartwy'] = pelny_kalendarz['JestMartwy'].astype(int)

In [108]:
braki = pelny_kalendarz.isna().sum()
print(braki[braki > 0])

Series([], dtype: int64)


In [103]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_sprzedaz (surowe): {len(df_sprzedaz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_sprzedaz)}")

print(f"\nRozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")


Wiersze w df_sprzedaz (surowe): 3,373,975
Realne rekordy w pelny_kalendarz: 3,373,975
Zgodność: True

Rozmiar: (8802515, 33)
Pamięć: 3395.0 MB

Kolumny: ['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow', 'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane', 'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwy']


In [104]:
pelny_kalendarz.to_parquet(
    "dane/interim/kalendarz_pelny_towid.parquet",
    compression='zstd',
    index=False
)


In [105]:
# Suma kontrolna
nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   kalendarz_pelny_towid.parquet   defac931291e6bad894c972b985ab043503fa4bb1ee673152bdcc24d27aa7c17


In [ ]:
# hash pliku kalendarz_pelny_towid.parquet:  defac931291e6bad894c972b985ab043503fa4bb1ee673152bdcc24d27aa7c17